In [1]:
from datetime import datetime, timedelta
import csv
import sqlite3
import requests
from bs4 import BeautifulSoup
import json
from typing import Dict, List
import re

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.express as px
import nbformat
import plotly.graph_objects as go

In [3]:
import geopandas as gpd
from shapely.geometry import Point

In [3]:
delay = pd.read_csv("Delay_2018_2019.csv") 

In [8]:
delay2 = pd.read_csv("Delay_2020_2021.csv") 

In [10]:
delay3 = pd.read_csv("Delay_2022_2023.csv") 

In [13]:
delay4 = pd.read_csv("Delay_2024.csv") 

In [14]:
delay4.head()

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,12,MQ,Envoy Air,EVV,"Evansville, IN: Evansville Regional",61.0,9.0,1.52,1.08,...,0.0,5.84,0.0,0.0,732.0,47.0,90.0,19.0,0.0,576.0
1,2024,12,MQ,Envoy Air,EWR,"Newark, NJ: Newark Liberty International",107.0,42.0,6.01,5.89,...,0.0,4.94,0.0,0.0,2531.0,335.0,491.0,1251.0,0.0,454.0
2,2024,12,MQ,Envoy Air,EYW,"Key West, FL: Key West International",169.0,31.0,3.37,0.71,...,0.0,15.48,5.0,3.0,1596.0,143.0,52.0,468.0,0.0,933.0
3,2024,12,MQ,Envoy Air,FAR,"Fargo, ND: Hector International",171.0,35.0,4.64,2.12,...,0.0,12.92,2.0,0.0,2428.0,245.0,184.0,575.0,0.0,1424.0
4,2024,12,MQ,Envoy Air,FSD,"Sioux Falls, SD: Joe Foss Field",69.0,14.0,2.00,2.47,...,0.0,4.83,1.0,0.0,720.0,86.0,154.0,191.0,0.0,289.0


In [11]:
delay.dropna(inplace=True)

In [5]:
delay.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44914 entries, 0 to 44913
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   year                 44914 non-null  int64  
 1   month                44914 non-null  int64  
 2   carrier              44914 non-null  object 
 3   carrier_name         44914 non-null  object 
 4   airport              44914 non-null  object 
 5   airport_name         44914 non-null  object 
 6   arr_flights          44862 non-null  float64
 7   arr_del15            44853 non-null  float64
 8   carrier_ct           44862 non-null  float64
 9   weather_ct           44862 non-null  float64
 10  nas_ct               44862 non-null  float64
 11  security_ct          44862 non-null  float64
 12  late_aircraft_ct     44862 non-null  float64
 13  arr_cancelled        44862 non-null  float64
 14  arr_diverted         44862 non-null  float64
 15  arr_delay            44862 non-null 

In [4]:
# ================ DATABASE SETUP =====================

# Create or connect to the database
conn = sqlite3.connect("Airline_delay.db")
conn.execute("PRAGMA foreign_keys = ON")  # Enforces foreign key constraints - execute function id=s needed to execute queries...
cursor = conn.cursor()  # this executes the query

In [7]:
# ================== Create a Product table ====================

cursor.execute("""     
CREATE TABLE IF NOT EXISTS Delays (
    year INTEGER,
    month INTEGER,
    carrier TEXT,
    carrier_name TEXT,
    airport TEXT,
    airport_name TEXT,
    arr_flights FLOAT,
    arr_del15 FLOAT,
    carrier_ct FLOAT,
    weather_ct FLOAT,
    nas_ct FLOAT,
    security_ct FLOAT,
    late_aircraft_ct FLOAT,
    arr_cancelled FLOAT,
    arr_diverted FLOAT,
    arr_delay FLOAT,
    carrier_delay FLOAT,
    weather_delay FLOAT,
    nas_delay FLOAT,
    security_delay FLOAT,
    late_aircraft_delay FLOAT
               
)
""")
conn.commit() #saves any changes made to the database permanently. 

In [17]:
delay.to_sql("Delays", conn, if_exists="append", index=False)

44853

In [9]:
delay2.to_sql("Delays", conn, if_exists="append", index=False)

40967

In [11]:
delay3.to_sql("Delays", conn, if_exists="append", index=False)

42077

In [15]:
delay4.to_sql("Delays", conn, if_exists="append", index=False)

20904

In [4]:
class DataLoader:
    def __init__(self, file_path):
        self.file_path = file_path

    def load_csv(self):
        import pandas as pd
        df = pd.read_csv(self.file_path)
        return df

    def load_multiple_files(self, file_list):
        import pandas as pd
        dfs = [pd.read_csv(f) for f in file_list]
        return pd.concat(dfs, ignore_index=True)

In [4]:
class DataCleaner:
    def clean(self, df):
        df = df.copy()

        # handle missing values
        #df.dropna(inplace=True)
        df = df.dropna(subset=['arr_delay'])

        #create features
        df['is_delayed'] = df['arr_delay'] > 15
        df['delay_minutes'] = df['arr_delay'].clip(lower=0)

        # extract time features
        #df['hour'] = df['CRSDepTime'] // 100
        #df['month'] = df['month']

        return df

In [4]:
class DatabaseManager:
    def __init__(self, db_name = "Airline_delay.db"):
        self.conn = sqlite3.connect(db_name)
        self.cursor = self.conn.cursor()

    #def save_table(self, df, table_name):
    #    df.to_sql(table_name, self.conn, if_exists='replace', index=False)

    def query(self, sql_query):
        return pd.read_sql(sql_query, self.conn)
    
    def add_column(self, table_name, column_name, data_type):
        self.cursor.execute(f"""
        ALTER TABLE {table_name}
        ADD COLUMN {column_name} {data_type};
        """)
        self.conn.commit()
    
    def execute2(self, sql_query):
        self.cursor.execute(sql_query)
        self.conn.commit()

    def execute(self, sql_query, params=None):
        if params:
            self.cursor.execute(sql_query, params)
        else:
            self.cursor.execute(sql_query)
        self.conn.commit()

    def close(self):
        self.conn.close()

In [6]:
class Analyzer:
    def __init__(self, df):
        self.df = df

    def airline_reliability(self):
        return (
            self.df.groupby('carrier_name')
            .agg(avg_delay=('arr_delay', 'mean'),
                 delay_rate=('is_delayed', 'mean'),
                 total_flights=('carrier', 'count'))
            .sort_values(by='delay_rate')
        )
    
    def delays_by_hour(self):
        return (
            self.df.groupby('hour')['arr_delay']
            .mean()
            .sort_index()
        )

    def airport_bottlenecks(self):
        return (
            self.df.groupby('airport')
            .agg(avg_delay=('arr_delay', 'mean'),
                 traffic=('airport', 'count'))
            .sort_values(by='avg_delay', ascending=False)
        )

In [7]:
class Visualizer:
    def plot_airline_performance(self, df):
        
        df['delay_rate'].plot(kind='bar')
        plt.title("Airline Delay Rate")
        plt.ylabel("Delay Rate")
        plt.show()

    def plot_delay_trends(self, df):
        
        df.plot()
        plt.title("Delays by Hour")
        plt.xlabel("Hour")
        plt.ylabel("Average Delay")
        plt.show()

In [5]:
#connect to DB
conn = sqlite3.connect("Airline_delay.db")
df = pd.read_sql("SELECT * FROM delays", conn)

In [6]:
db = DatabaseManager()

In [8]:
df['arr_delay'] = df['arr_delay'] / df['arr_flights']

In [11]:
df['arr_delay'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 148801 entries, 0 to 148800
Series name: arr_delay
Non-Null Count   Dtype  
--------------   -----  
148588 non-null  float64
dtypes: float64(1)
memory usage: 1.1 MB


In [11]:
airport = db.query("""
SELECT carrier_name, COUNT(carrier_name) AS total_flights FROM delays
                   WHERE year = 2018 and month = 12
                   GROUP BY carrier_name
                   ORDER BY total_flights desc
""")

print(airport.head(10))


                carrier_name  total_flights
0      SkyWest Airlines Inc.            237
1    Delta Air Lines Network            147
2                  Envoy Air            139
3          Endeavor Air Inc.            122
4              Allegiant Air            120
5   United Air Lines Network            103
6         Mesa Airlines Inc.            103
7   ExpressJet Airlines Inc.            102
8  American Airlines Network            102
9           Republic Airline             92


In [12]:
airport = db.query("""
SELECT  airport, AVG(arr_delay/arr_flights) AS sum_delay FROM delays
                   WHERE airport IN ('ORD','SFO') AND year = 2018
                   GROUP BY airport
""")

print(airport.head())


  airport  sum_delay
0     ORD  17.217138
1     SFO  16.644716


In [28]:
airport = db.query("""
SELECT  airport, SUM(arr_flights) AS avg FROM delays
                   WHERE airport IN ('ORD','SFO') AND year = 2018
                   GROUP BY airport
""")

print(airport.head())

  airport       avg
0     ORD  378881.0
1     SFO  171556.0


In [16]:
db = DatabaseManager()

result = db.query("""
SELECT * FROM delays WHERE 1=0
""")

print(result.head())

Empty DataFrame
Columns: [year, month, carrier, carrier_name, airport, airport_name, arr_flights, arr_del15, carrier_ct, weather_ct, nas_ct, security_ct, late_aircraft_ct, arr_cancelled, arr_diverted, arr_delay, carrier_delay, weather_delay, nas_delay, security_delay, late_aircraft_delay, state]
Index: []

[0 rows x 22 columns]


In [13]:
db = DatabaseManager()

result = db.query("""
SELECT month, AVG(arr_delay) AS avg_delay
FROM delays
WHERE year = 2021
GROUP BY month
ORDER BY month
""")

print(result)

    month    avg_delay
0       1  1393.732968
1       2  1917.666875
2       3  1629.289742
3       4  1836.963429
4       5  2465.359257
5       6  4985.643172
6       7  5829.352709
7       8  5001.215359
8       9  2616.229338
9      10  3562.358119
10     11  2679.790556
11     12  4356.448876
